# Notebook 03 — Covariance and Regime Modeling

**Purpose:** Build two covariance matrices — normal regime and stress regime. Capture how instrument yields move together and how that relationship changes during stress events.

**Inputs:** 
- `data/processed/expected_returns.csv`
- `data/raw/aave_tvl.csv`
- `data/raw/pendle_tvl.csv`

**Outputs:** 
- `data/processed/cov_normal.csv`
- `data/processed/cov_stress.csv`
- `data/processed/cov_blended.csv`

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(os.path.abspath('../src'))

import covariance as cov
import utils

np.random.seed(42)

## 1. Prepare Yield Series

Pivot the data to wide format (dates as index, instruments as columns).

In [ ]:
returns_df = pd.read_csv('../data/processed/expected_returns.csv')
pivot_df = returns_df.pivot(index='date', columns='instrument', values='yield_idr')
pivot_df = pivot_df.ffill().dropna()
pivot_df.head()

## 2. Classify Regimes

Based on TVL drops in Aave (>10%) or Pendle (>15%).

In [ ]:
aave_tvl = pd.read_csv('../data/raw/aave_tvl.csv', comment='#')
pendle_tvl = pd.read_csv('../data/raw/pendle_tvl.csv', comment='#')

# Ensure date overlap with pivot_df
aave_tvl = aave_tvl.set_index('date').reindex(pivot_df.index).ffill()
pendle_tvl = pendle_tvl.set_index('date').reindex(pivot_df.index).ffill()

regimes = cov.classify_regime(aave_tvl['tvl_usd'], pendle_tvl['tvl_usd'])
print(f"Stress days found: {(regimes == 'stress').sum()}")

## 3. Estimate Matrices

**Assumption:** If stress days < 30, we use a synthetic stress matrix as per the brief.

In [ ]:
cov_normal = cov.estimate_covariance(pivot_df, regimes, 'normal')
cov_stress = cov.estimate_covariance(pivot_df, regimes, 'stress')

if (regimes == 'stress').sum() < 30:
    print("Insufficient stress data. Using assumptions for stress covariance.")
    cov_stress = cov.construct_stress_covariance_assumption(cov_normal)

cov_blended = cov.blend_covariance(cov_normal, cov_stress, stress_prob=0.15)

cov_normal.to_csv('../data/processed/cov_normal.csv')
cov_stress.to_csv('../data/processed/cov_stress.csv')
cov_blended.to_csv('../data/processed/cov_blended.csv')

print("Covariance matrices saved successfully.")